# 1. QA 테스트셋 생성

## `(1) load pdf`

In [6]:
from langchain_community.document_loaders import PyPDFLoader

pdf_loader = PyPDFLoader('./pdf_files/채채봇 플랫폼 사내 매뉴얼 v1.2.pdf')
pdf_docs = pdf_loader.load()

len(pdf_docs)

12

## `(2) 문서 분할`

In [7]:
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer = tokenizer,
    chunk_size=300,        # 청크 크기  
    chunk_overlap=100,      # 청크 중 중복되는 부분 크기
)

chunks = text_splitter.split_documents(pdf_docs)

print(f"생성된 청크 수: {len(chunks)}")
print(f"각 청크의 길이: {list(len(chunk.page_content) for chunk in chunks)}")
print()

for chunk in chunks[:5]:
    tokens = tokenizer.encode(chunk.page_content) # 각 청크를 토큰화
    print(len(tokens)) # 각 청크의 단어 수 확인
    print(tokens[:10]) # 각 청크의 토큰화 결과 확인 (첫 10개 토큰만 출력)
    token_strings = tokenizer.convert_ids_to_tokens(tokens[:10]) # 토큰 ID를 실제 문자열로 변환해서 출력
    print(token_strings)
    print("=" * 50)
    print()

생성된 청크 수: 24
각 청크의 길이: [664, 265, 419, 361, 473, 315, 500, 495, 179, 492, 492, 388, 509, 498, 212, 590, 426, 583, 373, 588, 349, 612, 575, 444]

285
[0, 27235, 44115, 240229, 147035, 5939, 10459, 18692, 60600, 47321]
['<s>', '▁채', '채', '봇', '▁플랫폼', '▁사', '내', '▁매', '뉴', '얼']

116
[0, 87750, 171281, 152, 25763, 6292, 15, 670, 7710, 56]
['<s>', '▁문서', '▁소유', '▁:', '▁운영', '실', '▁(', 'O', 'wn', 'er']

219
[0, 27235, 32672, 152, 56750, 15, 94399, 5844, 247, 13964]
['<s>', '▁채', '널', '▁:', '▁웹', '▁(', 'Respons', 'ive', '),', '▁iOS']

272
[0, 7381, 43587, 6547, 1083, 787, 26157, 46039, 74523, 131091]
['<s>', '▁↑', '▁목', '차', '로', '▁2.', '▁제품', '▁안내', '▁주요', '▁카테고리']

295
[0, 16837, 81733, 248, 15852, 208647, 2085, 12, 16837, 12057]
['<s>', '▁양', '념', '▁/', '▁알', '뿌', '리', ':', '▁양', '파']



## `(3) 벡터저장소 저장`

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Hugging Face의 임베딩 모델 생성
embeddings_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model,
    collection_name="db_huggingface",
    persist_directory="./chroma",
    collection_metadata={"hnsw:space": "cosine"},
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## `(4) 메타데이터 확장 & JSONL 저장`

In [8]:
import os
import json

final_docs = []
for i, doc in enumerate(chunks):
    new_doc = doc.model_copy() 
    new_doc.metadata['doc_id'] = i # metadata에 doc id를 추가
    new_doc.page_content = str(new_doc.page_content).replace("[.!?]\\s+", "\n") # metadata에서 정보를 추출하여 page_content에 추가
    corp_name = str(os.path.split(new_doc.metadata['source'])[1].split('.')[0])
    new_doc.page_content = f"{new_doc.page_content}\n\n(참고: 이 문서는 {corp_name}에 대한 정보를 담고 있습니다.)"
    final_docs.append(new_doc)

In [9]:
with open('./data/final_docs.jsonl', 'wb') as f:
    for doc in final_docs:
        f.write(json.dumps(dict(doc)).encode('utf-8'))
        f.write(b'\n')

## `(5) Document 객체 변환 및 Dataframe 저장`

In [10]:
from langchain_community.document_loaders import JSONLoader

def metadata_func(record: dict, metadata: dict) -> dict:
    metadata = record.get("metadata")
    return metadata

json_loader = JSONLoader(
    file_path="./data/final_docs.jsonl",
    jq_schema=".",
    content_key="page_content",
    json_lines=True,
    metadata_func=metadata_func,
)

json_docs = json_loader.load()

In [11]:
import pandas as pd

test_data = []
for doc in json_docs:
    test_data.append({
        'context': str(doc.page_content),
        'source': str(doc.metadata.get('source', '')),
        'doc_id': str(doc.metadata.get('doc_id', '')),
    })

df_test = pd.DataFrame(test_data)
print(df_test.shape)
df_test.head()

(24, 3)


,context,source,doc_id
0,채채봇 플랫폼 사내 매뉴얼 v1.1 · 통합본\n원문 유지본 · 마지막 ...,./pdf_files/채채봇 플랫폼 사내 매뉴얼 v1.2.pdf,0
1,"문서 소유 : 운영실 (Owner), 각 파트 리뷰어 (Co-Owner)\n갱...",./pdf_files/채채봇 플랫폼 사내 매뉴얼 v1.2.pdf,1
2,"채널 : 웹 (Responsive), iOS/Android 앱 , 관리자 (Back...",./pdf_files/채채봇 플랫폼 사내 매뉴얼 v1.2.pdf,2
3,"↑ 목차로\n2. 제품 안내\n주요 카테고리\n잎채소: 상추 , 로메인 , 적상...",./pdf_files/채채봇 플랫폼 사내 매뉴얼 v1.2.pdf,3
4,"양념 / 알뿌리: 양파 , 샬롯 , 마늘 , 생강\n버섯류: 표고 , 느타리 , 새...",./pdf_files/채채봇 플랫폼 사내 매뉴얼 v1.2.pdf,4


## `(6) Question - Answer 합성`

In [ ]:
from pydantic import BaseModel, Field 
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain.schema import OutputParserException
from typing import List

# 질문-답변 쌍을 위한 Pydantic 모델 정의
class QAPair(BaseModel):
    """ 질문-답변 Pair"""
    question: str = Field(description="생성된 질문 (write your question in KOREAN)")
    answer: str = Field(description="질문에 대한 답변 (write the answer to the fact-based question in KOREAN, making sure it reflects the essence of the question)")

class QASet(BaseModel):
    qa_pairs: List[QAPair] = Field(default_factory=list, description="질문-답변 Pair의 리스트")


# QA 생성 템플릿
QA_generation_template = """
Your task is to create {num_questions_per_chunk} fact-based question-answer pairs based on the provided context.
Each fact-based question should be answerable with a specific, concise piece of factual information from the context.
Formulate your questions in the style that users might use when asking a search engine.
Avoid including phrases like "according to the passage" or "based on the context" in your questions.
Ensure that the answer includes the essence of the question to provide clear and complete information.
Write answers in a descriptive, paragraph style (about 2–4 full sentences) in KOREAN, avoiding one-word or terse replies.

---------------------------------------------------------
Provide your output in the following format:

{format_instructions}

---------------------------------------------------------
이제 컨텍스트를 제공합니다:

컨텍스트: {context}
"""

# ChatOpenAI 모델 초기화
qa_generator = ChatOllama(
    model = "exaone3.5",
    temperature=0.5,
)

# Pydantic 출력 파서 설정
pydantic_parser = PydanticOutputParser(pydantic_object=QASet)

# QA 생성 프롬프트 템플릿 생성
QA_generation_prompt = ChatPromptTemplate.from_template(
    template=QA_generation_template,
    partial_variables={"format_instructions": pydantic_parser.get_format_instructions()}
)

# QA 생성 체인 구성
QA_generate_chain = QA_generation_prompt | qa_generator | pydantic_parser

# 테스트 데이터셋 생성 함수
def generate_qa_dataset(context: str, num_questions: int) -> QASet:
    try:
        resp = QA_generate_chain.invoke({
            "context": context,
            "num_questions_per_chunk": num_questions
        })
        
        if resp is None:
            return QASet()  
        if isinstance(resp, QASet):
            return resp

        text = getattr(resp, "content", resp)
        if text is None or str(text).strip().lower() in ("", "null", "none"):
            return QASet()
        return pydantic_parser.parse(text)

    except OutputParserException:
        return QASet()
    except Exception:
        return QASet()


# QA 생성 테스트
test_context = df_test['context'][0]
print("컨텍스트:", test_context)
print()

qa_set = generate_qa_dataset(test_context, 2)
print("생성된 QA 쌍:")
for qa_pair in qa_set.qa_pairs:
    print(f"질문: {qa_pair.question}")
    print(f"답변: {qa_pair.answer}")
    print()

## `(7) 전체 Question - Answer 합성`

In [ ]:
NUM_QUESTIONS_PER_CHUNK = 3
outputs = []
for row in df_test.iterrows():

    qa_set = generate_qa_dataset(row[1]['context'], NUM_QUESTIONS_PER_CHUNK)

    if not qa_set.qa_pairs:
        print("빈 결과 → 스킵")
    else:
        for qa_pair in qa_set.qa_pairs:
            outputs.append({
                'context': [row[1]['context']],
                'source': [row[1]['source']],
                'doc_id': [row[1]['doc_id']],
                'question': qa_pair.question,
                'answer': qa_pair.answer
            })


df_qa_test = pd.DataFrame(outputs)
print(df_qa_test.shape)

df_qa_test.head()

## `(8) QA 테스트셋 엑셀 저장`

In [ ]:
df_qa_test.to_excel("./data/qa_testset.xlsx", index=False)

In [15]:
df_qa_test = pd.read_excel("./data/qa_testset.xlsx")
df_qa_test.head(20)

,context,source,doc_id,question,answer
0,['채채봇 플랫폼 사내 매뉴얼 v1.1 · 통합본\n원문 유지본 · 마지...,['./pdf_files/채채봇 플랫폼 사내 매뉴얼 v1.2.pdf'],['0'],채채봇 플랫폼 사내 매뉴얼의 최신 업데이트 날짜는 언제인가요?,채채봇 플랫폼 사내 매뉴얼의 마지막 업데이트 날짜는 2025년 9월 21일입니다.
1,['채채봇 플랫폼 사내 매뉴얼 v1.1 · 통합본\n원문 유지본 · 마지...,['./pdf_files/채채봇 플랫폼 사내 매뉴얼 v1.2.pdf'],['0'],이 매뉴얼이 적용되는 주요 시스템은 무엇인가요?,"이 매뉴얼은 웹 및 모바일 앱, 백오피스 시스템, 내부 챗봇, 재고 관리, 주문 처..."
2,['채채봇 플랫폼 사내 매뉴얼 v1.1 · 통합본\n원문 유지본 · 마지...,['./pdf_files/채채봇 플랫폼 사내 매뉴얼 v1.2.pdf'],['0'],매뉴얼의 문서 소유와 갱신 주기는 어떻게 되나요?,"매뉴얼의 소유는 운영실이 맡고 있으며, 최소 월 1회 정기적으로 갱신됩니다. 기능 ..."
3,"['문서 소유 : 운영실 (Owner), 각 파트 리뷰어 (Co-Owner)\...",['./pdf_files/채채봇 플랫폼 사내 매뉴얼 v1.2.pdf'],['1'],플랫폼의 주요 목표 중 하나는 무엇인가요?,"플랫폼의 주요 목표 중 하나는 유통 단계를 최소화하여 가격 경쟁력을 확보하고, 친환..."
4,"['문서 소유 : 운영실 (Owner), 각 파트 리뷰어 (Co-Owner)\...",['./pdf_files/채채봇 플랫폼 사내 매뉴얼 v1.2.pdf'],['1'],직거래를 통해 어떤 이점을 얻나요?,직거래를 통해 당일 수확한 농산물을 산지에서 바로 배송함으로써 신선도를 유지하고 소...
5,"['문서 소유 : 운영실 (Owner), 각 파트 리뷰어 (Co-Owner)\...",['./pdf_files/채채봇 플랫폼 사내 매뉴얼 v1.2.pdf'],['1'],변경 관리 프로세스에서 배포 전에 어떤 단계가 포함되어 있나요?,"변경 관리 프로세스에서 배포 전에 변경 요청(PR)을 제출하고, 최소 두 명 이상의..."
6,"['채널 : 웹 (Responsive), iOS/Android 앱 , 관리자 (Ba...",['./pdf_files/채채봇 플랫폼 사내 매뉴얼 v1.2.pdf'],['2'],채널은 어떤 플랫폼에서 사용 가능합니까?,"웹사이트(반응형 디자인 포함), iOS 및 Android 모바일 앱, 그리고 관리자..."
7,"['채널 : 웹 (Responsive), iOS/Android 앱 , 관리자 (Ba...",['./pdf_files/채채봇 플랫폼 사내 매뉴얼 v1.2.pdf'],['2'],고객이 상품을 구매할 때 주요 기능은 무엇입니까?,"고객은 회원가입 및 로그인, 상품 검색 및 구매, 실시간 재고 확인 및 배송 추적,..."
8,"['채널 : 웹 (Responsive), iOS/Android 앱 , 관리자 (Ba...",['./pdf_files/채채봇 플랫폼 사내 매뉴얼 v1.2.pdf'],['2'],사내 챗봇의 주요 목적은 무엇이며 어떤 시스템과 연동됩니까?,"사내 챗봇의 주요 목적은 주문 상태 확인, 재고 부족 알림 제공, 발주 프로세스 자..."
9,"['↑ 목차로\n2. 제품 안내\n주요 카테고리\n잎채소: 상추 , 로메인 , ...",['./pdf_files/채채봇 플랫폼 사내 매뉴얼 v1.2.pdf'],['3'],상추 외에 어떤 잎채소들이 주요 카테고리에 포함되어 있나요?,"로메인, 적상추, 시금치, 깻잎, 청경채, 케일, 양배추, 치커리, 루꼴라 등 다양..."


# 2. 검색기 구현

## `(1) LLM 정의 및 벡터저장소 로드`

In [22]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="exaone3.5",
    temperature=0.7,
)

In [17]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    embedding_function=embeddings_model,
    collection_name="db_huggingface",
    persist_directory="./chroma",
    collection_metadata={"hnsw:space": "cosine"},
)

## `(2) BM25 Retriever`

In [18]:
from krag.tokenizers import KiwiTokenizer
from krag.retrievers import KiWiBM25RetrieverWithScore

kiwi_tokenizer = KiwiTokenizer(model_type='knlm', typos='basic')

bm25_retriever = KiWiBM25RetrieverWithScore(
    documents=chunks,
    kiwi_tokenizer=kiwi_tokenizer, 
    k=8,
    threshold=0.0,
)

## `(3) Similarity Retriever`

In [20]:
similarity_retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.0, "k": 8},
)

## `(4) Ensemble Retriever`

In [21]:
from langchain.retrievers import EnsembleRetriever

ensemble_retrievers = [similarity_retriever, bm25_retriever]
ensemble_retriever = EnsembleRetriever(
    retrievers=ensemble_retrievers, 
    weights=[0.5, 0.5]
)

## `(5) Multi Query Retriever`

In [23]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multiquery_chroma_retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.0, "k": 8},
)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=multiquery_chroma_retriever, llm=llm
)

## `(6) Multi Query Custom Retriever`

In [24]:
from typing import List

from langchain_core.output_parsers import BaseOutputParser
from langchain_core.prompts import PromptTemplate


# 출력 파서: LLM 결과를 질문 리스트로 변환
class LineListOutputParser(BaseOutputParser[List[str]]):
    """Output parser for a list of lines."""

    def parse(self, text: str) -> List[str]:
        """Split the text into lines and remove empty lines."""
        return [line.strip() for line in text.strip().split("\n") if line.strip()]


# 쿼리 생성 프롬프트
QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""Generate three different versions of the given user question to retrieve relevant documents from a vector database. The goal is to reframe the question from various perspectives to overcome limitations of distance-based similarity search.

    The generated questions should have the following characteristics:
    1. Maintain the core intent of the original question but use different expressions or viewpoints.
    2. Include synonyms or related concepts where possible.
    3. Slightly broaden or narrow the scope of the question to potentially include diverse relevant information.

    Write each question on a new line and include only the questions.

    [Original question]
    {question}
    
    [Alternative questions]
    """,
)

# 멀티쿼리 체인 구성
multiquery_chain = QUERY_PROMPT | llm | LineListOutputParser()

In [ ]:
multi_query_custom_retriever = MultiQueryRetriever(
    retriever=multiquery_chroma_retriever, 
    llm_chain=multiquery_chain, # 멀티쿼리 체인
    parser_key="lines"            
)  

## `(7) Multi Query Decompostion Retriever`

In [26]:
from langchain.prompts import PromptTemplate

QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""You are an AI language model assistant. Your task is to decompose the given input question into multiple sub-questions. 
    The goal is to break down the input into a set of sub-problems/sub-questions that can be answered independently.

    Follow these guidelines to generate the sub-questions:
    1. Cover various aspects related to the core topic of the original question.
    2. Each sub-question should be specific, clear, and answerable independently.
    3. Ensure that the sub-questions collectively address all important aspects of the original question.
    4. Consider temporal aspects (past, present, future) where applicable.
    5. Formulate the questions in a direct and concise manner.

    [Input question] 
    {question}

    [Sub-questions (5)]
    """,
)

# 쿼리 생성 체인
decomposition_chain = QUERY_PROMPT | llm | LineListOutputParser()

In [27]:
# 다중 쿼리 검색기 생성
multi_query_decompostion_retriever = MultiQueryRetriever(
    retriever=multiquery_chroma_retriever,    
    llm_chain=decomposition_chain,   # 서브 질문 생성 체인
    parser_key="lines"               
)  

## `(8) Cross Encoder Reranker Retriever`

In [28]:
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
re_ranker = CrossEncoderReranker(model=model, top_n=3)

In [29]:
from langchain.retrievers import ContextualCompressionRetriever

cross_encoder_reranker_retriever = ContextualCompressionRetriever(
    base_compressor=re_ranker, 
    base_retriever=multi_query_custom_retriever,
)

## `(9) Embed Filter Compression Retriever`

In [30]:
from langchain.retrievers.document_compressors import EmbeddingsFilter

embeddings_filter = EmbeddingsFilter(embeddings=embeddings_model, similarity_threshold=0.5)

embed_filter_compression_retriever = ContextualCompressionRetriever(
    base_compressor=embeddings_filter,                            
    base_retriever=cross_encoder_reranker_retriever,              
)

## `(10) Compression Rerank Retriever`

In [ ]:
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_community.document_transformers import EmbeddingsRedundantFilter

# 중복 문서 제거
redundant_filter = EmbeddingsRedundantFilter(embeddings=embeddings_model)

# 쿼리와 관련성이 높은 문서만 필터링
relevant_filter = EmbeddingsFilter(
    embeddings=embeddings_model, similarity_threshold=0.5
)

# Re-ranking
re_ranker = CrossEncoderReranker(model=model, top_n=3)

pipeline_compressor = DocumentCompressorPipeline(
    transformers=[redundant_filter, relevant_filter, re_ranker]
)

pipeline_compression_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor,
    base_retriever=multi_query_custom_retriever,
)

## `(11) Compression LLM-Rerank Retriever`

In [ ]:
from langchain.retrievers.document_compressors import LLMListwiseRerank

# 중복 문서 제거
redundant_filter = EmbeddingsRedundantFilter(embeddings=embeddings_model)

# 쿼리와 관련성이 높은 문서만 필터링
relevant_filter = EmbeddingsFilter(
    embeddings=embeddings_model, similarity_threshold=0.5
)

re_ranker = LLMListwiseRerank.from_llm(llm, top_n=3)

pipeline_compressor = DocumentCompressorPipeline(
    transformers=[redundant_filter, relevant_filter, re_ranker]
)

llm_rerank_compression_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor,
    base_retriever=multi_query_custom_retriever,
)

# 3. 검색기 성능 평가

## `(1) BM25, Similarity, Ensemble 평가지표 생성`

In [ ]:
from krag.utils import evaluate_retrieval_at_K

retrievers = {
    'bm25': bm25_retriever,
    'similarity_score': similarity_retriever,
    'ensemble': ensemble_retriever,
}

print("k:", similarity_retriever.search_kwargs)

df_evaluation, df_evaluation_data = evaluate_retrieval_at_K(
    df_qa_test, 
    k=similarity_retriever.search_kwargs['k'],  
    retrievers=retrievers, 
    ensemble=False, 
    rouge_method='rouge2', 
    threshold=0.8)

In [ ]:
df_evaluation

## `(2) Multi Query, Multi Query Custom, Multi Query Decompostion 평가지표 생성`

In [ ]:
# 평가지표 계산
from krag.utils import evaluate_retrieval_at_K

retrievers = {
    'multi_query' : multi_query_retriever,
    'multi_query_custom' : multi_query_custom_retriever,
    'multi_query_decompostion' : multi_query_decompostion_retriever,
}

print("k:", similarity_retriever.search_kwargs)

df_evaluation, df_evaluation_data = evaluate_retrieval_at_K(
    df_qa_test, 
    k=similarity_retriever.search_kwargs['k'],  
    retrievers=retrievers, 
    ensemble=False, 
    rouge_method='rouge2', 
    threshold=0.8)

In [ ]:
df_evaluation

## `(3) Cross Encoder Reranker, Embed Filter Compression, Compression Rerank, LLM-Rerank Retriever 평가지표 생성`

In [ ]:
# 평가지표 계산
from krag.utils import evaluate_retrieval_at_K

retrievers = {
    'cross_encoder_reranker' : cross_encoder_reranker_retriever,
    'embed_filter_compression' : embed_filter_compression_retriever,
    'compression_rerank' : pipeline_compression_retriever,
    'llm-rerank-retriever' : llm_rerank_compression_retriever,
}

print("k:", similarity_retriever.search_kwargs)

df_evaluation, df_evaluation_data = evaluate_retrieval_at_K(
    df_qa_test, 
    k=similarity_retriever.search_kwargs['k'],  
    retrievers=retrievers, 
    ensemble=False, 
    rouge_method='rouge2', 
    threshold=0.8)

In [ ]:
df_evaluation